# 4.5 IC Data consistency checks

# Table of Contents

#### 01. Import libraries

#### 02. Import data

#### 03. Initial consistency check

#### 04. Mixed-type data

#### 05. Missing values

#### 06. Duplicates

#### 07. Export changed dataframe

#### 08. Task 4.5

#### Step 2 - Use .describe() function to investigate

#### Step 3 - Search for mixed-type data using a for-loop

#### Step 5 - Search for missing values

#### Step 7 - Check for duplicate values

#### Step 9 - Export data

# 01. Import libraries

In [3]:
# Import libraries
import pandas as pd
import numpy as np
import os

# 02. Import data

In [10]:
# Import orders.csv data from Original Data folder and the modified orders_wrangled.csv data from Prepared Data folder
# Define a path variable as a shortcut in the full folder search path.  Note the "r' .... '" syntax used.
path = r'C:\Users\dirk8\CareerFoundry Projects\03-2025 Instacart Basket Analysis'
df_prods = pd.read_csv(os.path.join(path, 'Data', 'Original Data', 'EX 4.3', 'products.csv'), index_col = False)
df_ords = pd.read_csv(os.path.join(path, 'Data', 'Prepared Data', 'orders_wrangled.csv'), index_col = False)

In [16]:
# Verify df_prods loaded as expected
df_prods.head(2)

,product_id,product_name,aisle_id,department_id,prices
0,1,Chocolate Sandwich Cookies,61,19,5.8
1,2,All-Seasons Salt,104,13,9.3


In [34]:
# Verify df_ords loaded as expected
df_ords.head(2)

,order_id,user_id,order_number,orders_day_of_week,order_hour_of_day,days_since_prior_order
0,2539329,1,1,2,8,NaN
1,2398795,1,2,3,7,15.0


# 03. Initial consistency check

In [36]:
# Show descriptive stats for df_ords to investigate Accuracy of columns in the df
df_ords.describe()

,order_id,user_id,order_number,orders_day_of_week,order_hour_of_day,days_since_prior_order
count,3.421083e+06,3.421083e+06,3.421083e+06,3.421083e+06,3.421083e+06,3.214874e+06
mean,1.710542e+06,1.029782e+05,1.715486e+01,2.776219e+00,1.345202e+01,1.111484e+01
std,9.875817e+05,5.953372e+04,1.773316e+01,2.046829e+00,4.226088e+00,9.206737e+00
min,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,8.552715e+05,5.139400e+04,5.000000e+00,1.000000e+00,1.000000e+01,4.000000e+00
50%,1.710542e+06,1.026890e+05,1.100000e+01,3.000000e+00,1.300000e+01,7.000000e+00
75%,2.565812e+06,1.543850e+05,2.300000e+01,5.000000e+00,1.600000e+01,1.500000e+01
max,3.421083e+06,2.062090e+05,1.000000e+02,6.000000e+00,2.300000e+01,3.000000e+01


In [40]:
# Comment: Initial data consistency check above for df_ords looks good for 3 right-side columns, nothing unusual

# 04. Mixed-type data

In [54]:
# Our dfs so far have no mixed-type columns, so for this Exercise 4.5 we create a df for mixed-type testing purposes
# Create a dataframe
df_test = pd.DataFrame()

In [56]:
# Then create a mixed-type column
df_test['mix'] = ['a', 'b', 1, True]

In [58]:
# Verify df with mixed-type column values was created as expected
df_test.head()

,mix
0,a
1,b
2,1
3,True


In [60]:
# Checking a df for mixed-type columns, using advanced Python syntax (which we will learn in later Exercises)

for col in df_test.columns.tolist():
    weird = (df_test[[col]].applymap(type) != df_test[[col]].iloc[0].apply(type)).any(axis = 1)
    if len (df_test[weird]) > 0:
        print(col)

mix


C:\Users\dirk8\AppData\Local\Temp\ipykernel_22740\2916675830.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  weird = (df_test[[col]].applymap(type) != df_test[[col]].iloc[0].apply(type)).any(axis = 1)


In [62]:
# The syntax above worked, however, the error code suggests it's outdated. This corrects it to updated syntax below.
# We are replacing ".applymap()" with ".map()"

for col in df_test.columns.tolist():
    weird = (df_test[[col]].map(type) != df_test[[col]].iloc[0].apply(type)).any(axis = 1)
    if len (df_test[weird]) > 0:
        print(col)

mix


In [64]:
# Change datatype for consistency, in this example to string
df_test['mix'] = df_test['mix'].astype('str')

In [72]:
# Check result
print(df_test['mix'].dtype)

object


In [74]:
# Comment: If we wanted to change column to an int64 data type (above), we would have substituted 'int64" for 'str'

# 05. Missing values

In [126]:
# Find missing values in the df_prods dataframe
# The .isnull() produces Boolean True (1) or False (0) values as checks the df columns record by record
df_prods.isnull().sum()

product_id        0
product_name     16
aisle_id          0
department_id     0
prices            0
dtype: int64

In [82]:
# Since product_name has 16 missing values, create a subset dataframe named df_nan that contains these alone, then show it
df_nan = df_prods[df_prods['product_name'].isnull() == True]
df_nan

,product_id,product_name,aisle_id,department_id,prices
33,34,NaN,121,14,12.2
68,69,NaN,26,7,11.8
115,116,NaN,93,3,10.8
261,262,NaN,110,13,12.1
525,525,NaN,109,11,1.2
1511,1511,NaN,84,16,14.3
1780,1780,NaN,126,11,12.3
2240,2240,NaN,52,1,14.2
2586,2586,NaN,104,13,12.4
3159,3159,NaN,126,11,13.1


In [86]:
# After looking at df_nan, we decide to create a new df that omits the 16 product records with missing product_name values
# Note that it's best practice to create a new df rather than alter the original df when omitting missing values
# Another best practice is to compare dimensions (with .shape function) of old df vs newly-changed df
# Start by checking dimensions of df_prods
df_prods.shape

(49693, 5)

In [88]:
# Create the new df as mentioned above.
# Use a .isnull() == False condition for the relevant column because you want only records that have non-missing values

df_prods_clean = df_prods[df_prods['product_name'].isnull() == False]

In [90]:
# Verify the new df dimensions, to compare with original df dimensions
df_prods_clean.shape

(49677, 5)

# 06. Duplicates

In [92]:
# Next, we look for full duplicates (e.g. all columns' values are the same) in df_prods_clean and show these
# The .duplicated() function helps with this.  We will create a subset df to store the duplicate records.
df_dups = df_prods_clean[df_prods_clean.duplicated()]
df_dups

,product_id,product_name,aisle_id,department_id,prices
462,462,Fiber 4g Gummy Dietary Supplement,70,11,4.8
18459,18458,Ranger IPA,27,5,9.2
26810,26808,Black House Coffee Roasty Stout Beer,27,5,13.4
35309,35306,Gluten Free Organic Peanut Butter & Chocolate ...,121,14,6.8
35495,35491,Adore Forever Body Wash,127,11,9.9


In [94]:
# Best practice is to drop these full duplicates and store the resulting df into a new df name.
# The .drop_duplicates() function helps with this, applied to a df.  Always check dimensions of relevant dfs before and after.
df_prods_clean.shape

(49677, 5)

In [96]:
df_prods_clean_no_dups = df_prods_clean.drop_duplicates()

In [98]:
# Check dimensions of altered new df to compare with old df
df_prods_clean_no_dups.shape

(49672, 5)

# 07. Export changed dataframe

In [164]:
# Export the new, cleaned (after consistency checks) df applying an intuitive name
# Note: Include a "index = False" argument to OMIT the index column in the outputted df .csv file
# That will avoid having to delete a redundant index column when importing the .csv file in a subsequent Notebook
df_prods_clean_no_dups.to_csv(os.path.join(path, 'Data', 'Prepared Data', 'products_checked.csv'), index = False)

# 08. Task 4.5

# Step 2 - Use .describe() function to investigate

In [109]:
# Run df.describe() function on df_ords dataframe
# Share in a markdown cell whether anything about the data looks off or warrants further investigation
df_ords.describe()

,order_id,user_id,order_number,orders_day_of_week,order_hour_of_day,days_since_prior_order
count,3.421083e+06,3.421083e+06,3.421083e+06,3.421083e+06,3.421083e+06,3.214874e+06
mean,1.710542e+06,1.029782e+05,1.715486e+01,2.776219e+00,1.345202e+01,1.111484e+01
std,9.875817e+05,5.953372e+04,1.773316e+01,2.046829e+00,4.226088e+00,9.206737e+00
min,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,8.552715e+05,5.139400e+04,5.000000e+00,1.000000e+00,1.000000e+01,4.000000e+00
50%,1.710542e+06,1.026890e+05,1.100000e+01,3.000000e+00,1.300000e+01,7.000000e+00
75%,2.565812e+06,1.543850e+05,2.300000e+01,5.000000e+00,1.600000e+01,1.500000e+01
max,3.421083e+06,2.062090e+05,1.000000e+02,6.000000e+00,2.300000e+01,3.000000e+01


# Maximum days_since_prior_order = 30 needs further investigation. Other numerical column stats look OK.

# Step 3 - Search for mixed-type data using a for-loop

In [114]:
# Check df_ords df for mixed-type data
# Note: I replace the Exercise's use of ".applymap()" with ".map()" to update Python syntax to current usage

for col in df_ords.columns.tolist():
    weird = (df_ords[[col]].map(type) != df_ords[[col]].iloc[0].apply(type)).any(axis = 1)
    if len (df_ords[weird]) > 0:
        print(col)

# Step 4

# No mixed-type data in df_ords because Step 3 script output blank, e.g. no mixed-type data columns revealed

# Step 5 - Search for missing values

In [128]:
# Step 5 Run a check for missing values in df_ords dataframe
# Report findings and propose explanation for any missing values discovered
df_ords.isnull().sum()

order_id                       0
user_id                        0
order_number                   0
orders_day_of_week             0
order_hour_of_day              0
days_since_prior_order    206209
dtype: int64

In [130]:
# df_ords dataframe has 206209 (a large number, proportionally) missing values for days_since_prior_order column
# Need to create a new subset df with all records that have these missing values in that column, to review and investigate
df_ords_missing = df_ords[df_ords['days_since_prior_order'].isnull() == True]
df_ords_missing

,order_id,user_id,order_number,orders_day_of_week,order_hour_of_day,days_since_prior_order
0,2539329,1,1,2,8,NaN
11,2168274,2,1,2,11,NaN
26,1374495,3,1,1,14,NaN
39,3343014,4,1,6,11,NaN
45,2717275,5,1,3,12,NaN
...,...,...,...,...,...,...
3420930,969311,206205,1,4,12,NaN
3420934,3189322,206206,1,3,18,NaN
3421002,2166133,206207,1,6,19,NaN
3421019,2227043,206208,1,1,15,NaN


In [132]:
# After reviewing df_ords_missing, we decide to create a new df that omits the 206209 records with last column missing values
# Note that we will NOT overwrite any data
# Idea here is to end up with 3 dataframes: original df, missing values subset df and non-missing values subset df
# Start by checking dimensions of df_ords (orginal df)
df_ords.shape

(3421083, 6)

In [134]:
# Next we create the new df for non-missing values (we already created the missing values df above)
# Use a .isnull() == False condition for the relevant column because we want only records with non-missing values
df_ords_non_missing = df_ords[df_ords['days_since_prior_order'].isnull() == False]

In [136]:
# Verify the new df dimensions, to compare with original df dimensions
df_ords_non_missing.shape

(3214874, 6)

In [140]:
# As per above, df_ords and df_ords_non_missing have a difference of 206209 rows, matching df_ords_missing row count. Good.

# The df_ords dataframe has 206209 missing values for column 'days_since_prior_order'. These are likely for the first order of a new customer, because new customers' first order would not have a prior order off which to calculate a time differential. An additional, separate question is whether any of these missing values are for instances where the time gap between orders exceeds 30 days, since it was "weird" that the .describe() function stats had a maximum value of 30 days for the 'days_since_prior_order' column.

# Step 6

# In Step 5 I left the df_ords df (original data) untouched. Created 2 new subset dfs, one with the missing value records only, and a separate one with the non-missing value records. Assuming the "new customer first order" explanation is correct, we may need up to all 3 dfs for different analytical purposes. The tentative explanation for the missing values needs to be confirmed.

# Step 7 - Check for duplicate values

In [150]:
# Step 7 Run a check for duplicate values in the df_ords df
# In a Markdown cell, report findings and propose an explanation for any duplicate values found
# Note: I will look for full duplicates (all column values the same) via the .duplicated() function
df_ords_dups = df_ords[df_ords.duplicated()]
df_ords_dups

,order_id,user_id,order_number,orders_day_of_week,order_hour_of_day,days_since_prior_order


# No full duplicates records were found in the original dataframe df_ords. This also means the 2 subset dfs created previously with missing value and non-missing value records do not need a full duplicates check, as they are subsets of df_ords.

# Step 8

In [158]:
# Step 8 Address the duplicates using an appropriate method
# In a Markdown cell, explain why you used your method of choice

# There are no full duplicates to remove.  If there were, I would have created new subset dfs with non-duplicate records for all orders dfs affected. The 3 orders dfs in this Task 4.5 are now df_ords (original data), df_ords_missing (a subset df with column 'days_since_prior_order' missing values records) and df_ords_non_missing (a subset df excluding those records with the missing values).

# Step 9 - Export data

In [166]:
# Export final, cleaned df_prods and df_ords data as .csv files in 'Prepared Data' folder, use succinct names
# NOTE: The cleaned df_prods dataframe was already exported earlier (above) at the end of Exercise 4.5
# Note: I will include a "index = False" argument to OMIT the index column in the outputted .csv files
# This will avoid having to delete a redundant index column when importing the .csv files in a subsequent Notebook
# I will export df_ords and its 2 subset dfs in case up to all 3 are needed in future analysis goals
# Which one(s) will be needed later depends on the actual reason for the 'days_since_prior_order' column missing values, and analysis goals

In [169]:
df_ords.to_csv(os.path.join(path, 'Data', 'Prepared Data', 'orders_checked.csv'), index = False)

In [171]:
df_ords_missing.to_csv(os.path.join(path, 'Data', 'Prepared Data', 'orders_checked_missing.csv'), index = False)

In [168]:
df_ords_non_missing.to_csv(os.path.join(path, 'Data', 'Prepared Data', 'orders_checked_non_missing.csv'), index = False)